<a href="https://colab.research.google.com/github/PattaraphonD/Data-Science-Projects/blob/main/twitter_proj_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install tweepy

In [9]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import json

# Path to the secrets file
secrets_path = "/content/drive/MyDrive/twitter_proj/tw_api"

# Load API keys
with open(secrets_path, "r") as file:
    secrets = json.load(file)

# Assign secrets to variables
consumer_key = secrets["consumer_key"]
consumer_secret = secrets["consumer_secret"]
access_token = secrets["access_token"]
access_token_secret = secrets["access_token_secret"]
bearer_token = secrets["bearer_token"]

print("Secrets loaded successfully (hidden for security).")

Secrets loaded successfully (hidden for security).


In [5]:
import tweepy

# Authenticate using OAuth 1.0a
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth)

# Test: Get your Twitter username
user = api.verify_credentials()
print(f"Authenticated as {user.screen_name}")

Authenticated as Patt45168


In [7]:
# Define search query correctly
search_query = "food -filter:retweets -filter:replies"

# Number of tweets to fetch
no_of_tweets = 10

try:
    # Fetch tweets
    tweets = api.search_tweets(q=search_query, lang="en", count=no_of_tweets, tweet_mode='extended')

    # Extract tweet attributes
    attributes_container = [
        [tweet.user.name, tweet.created_at, tweet.favorite_count, tweet.source, tweet.full_text]
        for tweet in tweets
    ]

    # Column names
    columns = ["User", "Date Created", "Number of Likes", "Source of Tweet", "Tweet"]

    # Create DataFrame
    tweets_df = pd.DataFrame(attributes_container, columns=columns)

except BaseException as e:
    print('Status Failed On,',str(e))

Status Failed On, 403 Forbidden
453 - You currently have access to a subset of X API V2 endpoints and limited v1.1 endpoints (e.g. media post, oauth) only. If you need access to this endpoint, you may need a different access level. You can learn more here: https://developer.x.com/en/portal/product


In [32]:
import tweepy
import pandas as pd
import os

# Set up API v2 authentication
client = tweepy.Client(bearer_token=bearer_token)

# Function to fetch tweets safely
def fetch_tweets(query, max_results=10):
    tweets = client.search_recent_tweets(
        query=query,
        max_results=max_results,
        tweet_fields=["created_at", "public_metrics", "source"],
        expansions=["author_id"],
        user_fields=["username", "name", "location", "public_metrics"]
    )

    if not tweets.data:
        print("❌ No tweets found!")
        return None

    # Create a dictionary of users (user_id → user details)
    user_dict = {user.id: user for user in tweets.includes["users"]} if "users" in tweets.includes else {}

    # Extract tweet data
    tweet_list = []
    for tweet in tweets.data:
        user = user_dict.get(tweet.author_id, None)  # Get user details safely
        tweet_list.append([
            tweet.text,
            tweet.created_at,
            tweet.public_metrics["like_count"],
            tweet.public_metrics["retweet_count"],
            tweet.public_metrics["reply_count"],
            tweet.public_metrics["impression_count"],
            tweet.source,
            user.username if user else "Unknown",  # Handle missing users
            user.name if user else "Unknown",
            user.location if user and "location" in user else "N/A",
            user.public_metrics["followers_count"] if user else 0
        ])

    return pd.DataFrame(tweet_list, columns=["Tweet", "Date Created", "Likes", "Retweets", "Replies", "Views", "Source", "Username", "Full Name", "Location", "Followers"])

In [33]:
# Save to CSV function
def save_to_csv(df, filename="tweets.csv"):
    if df is None or df.empty:
        print("❌ No data to save!")
        return

    # Check if file exists
    file_exists = os.path.isfile(filename)

    # Append to file if it exists, otherwise write new file
    df.to_csv(filename, mode='a', header=not file_exists, index=False, encoding='utf-8-sig')

    print(f"✅ Tweets saved to {filename}")

In [35]:
# Fetch and save tweets
tweets_df = fetch_tweets(query, max_results=10)
save_to_csv(tweets_df, filename)

✅ Tweets saved to tweets.csv


In [41]:
tweets_df.to_csv("/content/drive/MyDrive/twitter_proj/twittertweets.csv", index=False, encoding="utf-8-sig")
print("✅ CSV uploaded to Google Drive!")

✅ CSV uploaded to Google Drive!


In [36]:
tweets_df

,Tweet,Date Created,Likes,Retweets,Replies,Views,Source,Username,Full Name,Location,Followers
0,Jos Buttler is not England's captain anymore 😮...,2025-02-28 14:28:40+00:00,1,0,1,274,None,tjml,Tajamul Adil,"Riyadh, Saudi Arabia",2700
1,#LISA - 'ALTER EGO' on Apple Music:\n\n#1 Mada...,2025-02-28 14:25:19+00:00,39,12,0,359,None,BPGlobalNews,BLACKPINK Global News,In Your Area (Since 2020),69469
2,FUTW might be my favorite from Lisa's Alter Eg...,2025-02-28 14:24:54+00:00,0,0,0,21,None,Shaggy092317,Sré,N/A,36
3,#jensoo has entered the #alterego https://t.co...,2025-02-28 14:24:42+00:00,0,0,0,4,None,BayAreaBlink,R,N/A,19
4,"Ahora sí, top de ALTEREGO \n1. New woman FT Ro...",2025-02-28 14:24:42+00:00,0,0,0,17,None,Dtdoong9,dan⁹ 🍒| 🇫🇮 ALTEREGO,Taylor's versión,1437
5,I thought I've already figured out what my top...,2025-02-28 14:24:22+00:00,1,0,0,8,None,jowaofLisa,blank,lilieland,409
6,🎧 ¡Viernes de #NovedadesconRITMO en @SpotifySp...,2025-02-28 14:24:00+00:00,0,0,0,45,None,conRITMOes,RITMO,N/A,17
7,Hi @LISANATIONS_ can you help informing @weare...,2025-02-28 14:22:22+00:00,0,0,0,8,None,liliemage,LILIE MAGEˢᵖᵉᵉᵈⁱ,Philippines,555
8,#review ALTEREGO\n\n🏅 Lifestyle\nNew Woman\nCh...,2025-02-28 14:21:27+00:00,0,0,1,2,None,hantisccial,headbangeeeer,free derry,64
9,Lowkey flipping over #ALTEREGO Rapunzel’s fuck...,2025-02-28 14:21:13+00:00,0,0,0,4,None,dirtydiana747,KIM WEXLER,+ Inside Taemin’s Move album +,145


In [40]:
df = pd.read_csv("tweets.csv")
print(df.head())  # Show first 5 rows

                                               Tweet  \
0  Jos Buttler is not England's captain anymore 😮...   
1  #LISA - 'ALTER EGO' on Apple Music:\n\n#1 Mada...   
2  FUTW might be my favorite from Lisa's Alter Eg...   
3  #jensoo has entered the #alterego https://t.co...   
4  Ahora sí, top de ALTEREGO \n1. New woman FT Ro...   

                Date Created  Likes  Retweets  Replies  Views  Source  \
0  2025-02-28 14:28:40+00:00      1         0        1    274     NaN   
1  2025-02-28 14:25:19+00:00     39        12        0    359     NaN   
2  2025-02-28 14:24:54+00:00      0         0        0     21     NaN   
3  2025-02-28 14:24:42+00:00      0         0        0      4     NaN   
4  2025-02-28 14:24:42+00:00      0         0        0     17     NaN   

       Username              Full Name                   Location  Followers  
0          tjml           Tajamul Adil       Riyadh, Saudi Arabia       2700  
1  BPGlobalNews  BLACKPINK Global News  In Your Area (Since 2020) 